In [ ]:
%cd /content
![ -d jumbax ] || git clone --branch thesis https://github.com/Griger5/jumbax.git
%cd /content/jumbax
!git checkout thesis
!git pull --ff-only
!git log -1 --format="%h %ad %s" --date=short

In [ ]:
!pip install -q "jumbax[mpdata-jax]" netCDF4 xarray tqdm

In [ ]:
import json
import time
import copy
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
import xarray as xr
from tqdm import tqdm

from jumbax_api.mpdata import setup_mpdata, solve_mpdata
from jumbax_core.common import Dtype
from jumbax_core.mpdata import BoundaryCondition

REPO_ROOT = Path.cwd()
BASE_DIR = REPO_ROOT / "benchmarks" / "tpu_google_colab"
DATA_DIR = REPO_ROOT / "benchmarks" / "implementations_comparison" / "data"
RESULTS_DIR = REPO_ROOT / "benchmarks" / "results"

with open(BASE_DIR / "config.json") as f:
    config = json.load(f)
iters = config["iters"]

available = {d.platform for d in jax.devices()}
if "tpu" in available:
    try:
        jax.distributed.initialize()
    except Exception:
        pass
    available = {d.platform for d in jax.devices()}

if "tpu" in available:
    dev = "tpu"
elif "gpu" in available:
    dev = "gpu"
else:
    dev = "cpu"
device = jax.devices(dev)[0]
print(f"Running {dev}: {device.device_kind}")

results_path = RESULTS_DIR / f"tpu_google_colab_{dev}.json"
results = {}

In [ ]:
data_files = sorted(DATA_DIR.glob(config["data"]), key=lambda p: (int(p.stem.split("_")[1]), int(p.stem.split("_")[2])))

for data_path in tqdm(data_files, position=0):
    ds = xr.open_dataset(data_path)
    data = (ds["psi"].to_numpy(), ds["Cx"].to_numpy(), ds["Cy"].to_numpy())
    metadata = {k: int(ds.attrs[k]) for k in ("size_x", "size_y", "halo", "steps", "n_iters")}

    entry = results.setdefault(data_path.name, {"metadata": metadata, "data": {}})
    entry["metadata"] = metadata

    h = metadata["halo"]
    sx, sy = metadata["size_x"], metadata["size_y"]
    shape = (sx + 2 * h, sy + 2 * h)

    with jax.default_device(device), setup_mpdata(
            "jax", n_dims=2, n_iters=metadata["n_iters"], epsilon=1e-12,
            dtype=Dtype.float32, halo=h,
            boundary_conditions=[BoundaryCondition.PERIODIC, BoundaryCondition.PERIODIC],
            mpi=False, n_threads=None):

        # Warmup outside the timer
        d = copy.deepcopy(data)
        psi = jnp.zeros(shape, dtype=jnp.float32).at[h:h + sx, h:h + sy].set(d[0])
        np.asarray(solve_mpdata(psi, (d[1], d[2]), metadata["steps"])[h:-h, h:-h])

        times = []
        for _ in range(iters):
            d = copy.deepcopy(data)
            psi = jnp.zeros(shape, dtype=jnp.float32).at[h:h + sx, h:h + sy].set(d[0])

            start = time.perf_counter()
            res = solve_mpdata(psi, (d[1], d[2]), metadata["steps"])
            res = np.asarray(res[h:-h, h:-h])
            times.append(time.perf_counter() - start)

    entry["data"][f"jax_mpdata_{dev}"] = {
        "device": device.device_kind,
        "times": tuple(float(f"{t:.3g}") for t in times),
    }
    tqdm.write(f"{data_path.stem} / {dev}: min={min(times):.6f}s avg={sum(times)/len(times):.6f}s")

    RESULTS_DIR.mkdir(exist_ok=True)
    with open(results_path, "w", encoding="UTF-8") as f:
        json.dump(results, f, sort_keys=True, indent=4)

In [ ]:
from google.colab import files

files.download(str(results_path))

expected = {"cpu", "gpu", "tpu"}
present = {p.stem.removeprefix("tpu_google_colab_") for p in RESULTS_DIR.glob("tpu_google_colab_*.json")}

if expected <= present:
    merged = {}
    for path in sorted(RESULTS_DIR.glob("tpu_google_colab_*.json")):
        for name, entry in json.loads(path.read_text()).items():
            target = merged.setdefault(name, {"metadata": entry["metadata"], "data": {}})
            target["data"].update(entry["data"])

    merged_path = RESULTS_DIR / "tpu_google_colab.json"
    merged_path.write_text(json.dumps(merged, sort_keys=True, indent=4))
    files.download(str(merged_path))

    columns = sorted({k for e in merged.values() for k in e["data"]})
    print("merged", len(merged), "grids")
    print("min seconds".ljust(26), " ".join(c.rjust(16) for c in columns))
    for name in sorted(merged, key=lambda n: (merged[n]["metadata"]["size_x"], merged[n]["metadata"]["n_iters"])):
        row = merged[name]["data"]
        print(name.ljust(26), " ".join(format(min(row[c]["times"]), ".6g").rjust(16) for c in columns))
else:
    print("waiting on:", sorted(expected - present))